# InsightForge: AI-Powered Business Intelligence Assistant

## Advanced Generative AI Project

This notebook implements an AI-powered Business Intelligence Assistant using:
- **LangChain** for LLM orchestration
- **RAG (Retrieval-Augmented Generation)** for context-aware responses
- **Memory Integration** for conversational context
- **QAEvalChain** for model evaluation
- **Streamlit** for user interface

### Project Objectives:
1. Analyze business data to identify key trends and patterns
2. Generate actionable insights and recommendations using NLP
3. Visualize data insights for easier interpretation

---
## Environment Setup

Install and import all required dependencies for the project.

In [1]:
!pip install langchain langchain-openai langchain-community langchain-huggingface faiss-cpu python-dotenv pandas matplotlib seaborn streamlit -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
!pip uninstall sentence-transformers -y && pip install sentence-transformers --force-reinstall --no-cache-dir

In [2]:
import os
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain, RetrievalQA, LLMChain
from langchain.evaluation.qa import QAEvalChain

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY not found. Create a .env file with your key "
        "or export it as an environment variable."
    )

# -- Plotting defaults
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# -- Configuration
DATA_PATH = Path("sales_data.csv")
LLM_MODEL = "gpt-3.5-turbo"
LLM_TEMPERATURE = 0
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RAG_TOP_K = 5

In [3]:
try:
    llm = ChatOpenAI(temperature=LLM_TEMPERATURE, model=LLM_MODEL)
    llm.invoke("ping")  # fast connectivity check
    logger.info("LLM initialised: %s", LLM_MODEL)
except Exception as exc:
    raise RuntimeError(
        f"Failed to initialise LLM ({LLM_MODEL}). "
        "Verify your OPENAI_API_KEY and network connection."
    ) from exc

---
# Part 1: AI-Powered Business Intelligence Assistant

## Step 1: Data Preparation

Load and explore the sales dataset to understand its structure and generate descriptive statistics.

In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found at '{DATA_PATH.resolve()}'")

df = pd.read_csv(DATA_PATH)

REQUIRED_COLS = {"Date", "Product", "Region", "Sales",
                 "Customer_Age", "Customer_Gender", "Customer_Satisfaction"}
missing = REQUIRED_COLS - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.to_period('M')
df['Quarter'] = df['Date'].dt.to_period('Q')
df['Year'] = df['Date'].dt.year

logger.info("Loaded %d rows from %s", len(df), DATA_PATH)

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
df.head(10)

Dataset Shape: (2500, 10)

Columns: ['Date', 'Product', 'Region', 'Sales', 'Customer_Age', 'Customer_Gender', 'Customer_Satisfaction', 'Month', 'Quarter', 'Year']

Data Types:
Date                     datetime64[ns]
Product                          object
Region                           object
Sales                             int64
Customer_Age                      int64
Customer_Gender                  object
Customer_Satisfaction           float64
Month                         period[M]
Quarter                   period[Q-DEC]
Year                              int32
dtype: object


,Date,Product,Region,Sales,Customer_Age,Customer_Gender,Customer_Satisfaction,Month,Quarter,Year
0,2022-01-01,Widget C,South,786,26,Male,2.874407,2022-01,2022Q1,2022
1,2022-01-02,Widget D,East,850,29,Male,3.365205,2022-01,2022Q1,2022
2,2022-01-03,Widget A,North,871,40,Female,4.547364,2022-01,2022Q1,2022
3,2022-01-04,Widget C,South,464,31,Male,4.555420,2022-01,2022Q1,2022
4,2022-01-05,Widget C,South,262,50,Female,3.982935,2022-01,2022Q1,2022
5,2022-01-06,Widget D,East,147,35,Female,1.140628,2022-01,2022Q1,2022
6,2022-01-07,Widget A,North,610,66,Male,3.300282,2022-01,2022Q1,2022
7,2022-01-08,Widget A,South,428,58,Male,4.146334,2022-01,2022Q1,2022
8,2022-01-09,Widget C,West,939,26,Male,1.069152,2022-01,2022Q1,2022
9,2022-01-10,Widget B,West,215,40,Male,3.198738,2022-01,2022Q1,2022


### Descriptive Statistics

In [5]:
print("=== Numerical Statistics ===")
print(df[['Sales', 'Customer_Age', 'Customer_Satisfaction']].describe())

print("\n=== Categorical Value Counts ===")
print(f"\nProducts:\n{df['Product'].value_counts()}")
print(f"\nRegions:\n{df['Region'].value_counts()}")
print(f"\nGender:\n{df['Customer_Gender'].value_counts()}")

=== Numerical Statistics ===
             Sales  Customer_Age  Customer_Satisfaction
count  2500.000000   2500.000000            2500.000000
mean    553.288000     43.332800               3.025869
std     260.101758     14.846758               1.156981
min     100.000000     18.000000               1.005422
25%     324.750000     31.000000               2.056014
50%     552.500000     43.000000               3.049480
75%     779.000000     56.000000               4.042481
max     999.000000     69.000000               4.999006

=== Categorical Value Counts ===

Products:
Product
Widget A    656
Widget C    620
Widget D    612
Widget B    612
Name: count, dtype: int64

Regions:
Region
West     644
North    639
South    628
East     589
Name: count, dtype: int64

Gender:
Customer_Gender
Female    1256
Male      1244
Name: count, dtype: int64


In [6]:
print("=== Sales Performance Metrics ===")
print(f"Total Sales: ${df['Sales'].sum():,.2f}")
print(f"Average Sale: ${df['Sales'].mean():,.2f}")
print(f"Median Sale: ${df['Sales'].median():,.2f}")
print(f"Sales Std Dev: ${df['Sales'].std():,.2f}")

print("\n=== Customer Metrics ===")
print(f"Average Customer Age: {df['Customer_Age'].mean():.1f} years")
print(f"Average Satisfaction: {df['Customer_Satisfaction'].mean():.2f}/5")

print("\n=== Date Range ===")
print(f"From: {df['Date'].min().strftime('%Y-%m-%d')}")
print(f"To: {df['Date'].max().strftime('%Y-%m-%d')}")

=== Sales Performance Metrics ===
Total Sales: $1,383,220.00
Average Sale: $553.29
Median Sale: $552.50
Sales Std Dev: $260.10

=== Customer Metrics ===
Average Customer Age: 43.3 years
Average Satisfaction: 3.03/5

=== Date Range ===
From: 2022-01-01
To: 2028-11-04


---
## Step 2: Knowledge Base Creation

Create text summaries and document chunks from the data for the RAG system. This organizes the data into a structured format suitable for retrieval and analysis.

In [7]:
def create_data_summary(df: pd.DataFrame) -> list[Document]:
    """Generate comprehensive text summaries of the sales data for RAG.

    Creates one Document per logical slice (overall, per-product, per-region,
    per-month, per-age-group, per-gender) so the vector store can retrieve
    the most relevant context for a given query.
    """
    documents: list[Document] = []
    
    overall_summary = f"""
    OVERALL BUSINESS SUMMARY:
    The dataset contains {len(df)} sales transactions from {df['Date'].min().strftime('%B %Y')} to {df['Date'].max().strftime('%B %Y')}.
    Total Revenue: ${df['Sales'].sum():,.2f}
    Average Transaction Value: ${df['Sales'].mean():.2f}
    Median Transaction Value: ${df['Sales'].median():.2f}
    Standard Deviation: ${df['Sales'].std():.2f}
    Products Sold: {', '.join(df['Product'].unique())}
    Regions Covered: {', '.join(df['Region'].unique())}
    Customer Age Range: {df['Customer_Age'].min()} to {df['Customer_Age'].max()} years
    Average Customer Satisfaction: {df['Customer_Satisfaction'].mean():.2f} out of 5
    """
    documents.append(Document(page_content=overall_summary, metadata={"type": "overall_summary"}))
    
    for product in df['Product'].unique():
        product_df = df[df['Product'] == product]
        product_summary = f"""
        PRODUCT ANALYSIS - {product}:
        Total Sales: ${product_df['Sales'].sum():,.2f}
        Number of Transactions: {len(product_df)}
        Average Sale: ${product_df['Sales'].mean():.2f}
        Median Sale: ${product_df['Sales'].median():.2f}
        Best Performing Region: {product_df.groupby('Region')['Sales'].sum().idxmax()}
        Average Customer Satisfaction: {product_df['Customer_Satisfaction'].mean():.2f}
        Primary Customer Gender: {product_df['Customer_Gender'].value_counts().idxmax()}
        Average Customer Age: {product_df['Customer_Age'].mean():.1f} years
        """
        documents.append(Document(page_content=product_summary, metadata={"type": "product", "product": product}))
    
    for region in df['Region'].unique():
        region_df = df[df['Region'] == region]
        region_summary = f"""
        REGIONAL ANALYSIS - {region} Region:
        Total Sales: ${region_df['Sales'].sum():,.2f}
        Number of Transactions: {len(region_df)}
        Average Sale: ${region_df['Sales'].mean():.2f}
        Top Selling Product: {region_df.groupby('Product')['Sales'].sum().idxmax()}
        Average Customer Satisfaction: {region_df['Customer_Satisfaction'].mean():.2f}
        Gender Distribution: Male {len(region_df[region_df['Customer_Gender']=='Male'])}, Female {len(region_df[region_df['Customer_Gender']=='Female'])}
        Average Customer Age: {region_df['Customer_Age'].mean():.1f} years
        """
        documents.append(Document(page_content=region_summary, metadata={"type": "region", "region": region}))
    
    for month in df['Month'].unique():
        month_df = df[df['Month'] == month]
        month_summary = f"""
        MONTHLY ANALYSIS - {month}:
        Total Sales: ${month_df['Sales'].sum():,.2f}
        Number of Transactions: {len(month_df)}
        Average Sale: ${month_df['Sales'].mean():.2f}
        Best Selling Product: {month_df.groupby('Product')['Sales'].sum().idxmax()}
        Best Performing Region: {month_df.groupby('Region')['Sales'].sum().idxmax()}
        Average Satisfaction: {month_df['Customer_Satisfaction'].mean():.2f}
        """
        documents.append(Document(page_content=month_summary, metadata={"type": "monthly", "month": str(month)}))
    
    age_bins = [18, 30, 45, 60, 100]
    age_labels = ['Young (18-29)', 'Middle (30-44)', 'Senior (45-59)', 'Elder (60+)']
    df['Age_Group'] = pd.cut(df['Customer_Age'], bins=age_bins, labels=age_labels, right=False)
    
    for age_group in age_labels:
        age_df = df[df['Age_Group'] == age_group]
        if len(age_df) > 0:
            age_summary = f"""
            CUSTOMER SEGMENT - {age_group}:
            Number of Customers: {len(age_df)}
            Total Sales: ${age_df['Sales'].sum():,.2f}
            Average Sale: ${age_df['Sales'].mean():.2f}
            Preferred Product: {age_df.groupby('Product')['Sales'].sum().idxmax()}
            Average Satisfaction: {age_df['Customer_Satisfaction'].mean():.2f}
            Gender Split: Male {len(age_df[age_df['Customer_Gender']=='Male'])}, Female {len(age_df[age_df['Customer_Gender']=='Female'])}
            """
            documents.append(Document(page_content=age_summary, metadata={"type": "customer_segment", "segment": age_group}))
    
    for gender in df['Customer_Gender'].unique():
        gender_df = df[df['Customer_Gender'] == gender]
        gender_summary = f"""
        GENDER ANALYSIS - {gender} Customers:
        Number of Transactions: {len(gender_df)}
        Total Sales: ${gender_df['Sales'].sum():,.2f}
        Average Sale: ${gender_df['Sales'].mean():.2f}
        Preferred Product: {gender_df.groupby('Product')['Sales'].sum().idxmax()}
        Average Age: {gender_df['Customer_Age'].mean():.1f} years
        Average Satisfaction: {gender_df['Customer_Satisfaction'].mean():.2f}
        Top Region: {gender_df.groupby('Region')['Sales'].sum().idxmax()}
        """
        documents.append(Document(page_content=gender_summary, metadata={"type": "gender", "gender": gender}))
    
    return documents

knowledge_base_docs = create_data_summary(df)
print(f"Created {len(knowledge_base_docs)} documents for the knowledge base")

Created 98 documents for the knowledge base


In [8]:
print("=== Sample Document (Overall Summary) ===")
print(knowledge_base_docs[0].page_content)
print(f"\nMetadata: {knowledge_base_docs[0].metadata}")

=== Sample Document (Overall Summary) ===

    OVERALL BUSINESS SUMMARY:
    The dataset contains 2500 sales transactions from January 2022 to November 2028.
    Total Revenue: $1,383,220.00
    Average Transaction Value: $553.29
    Median Transaction Value: $552.50
    Standard Deviation: $260.10
    Products Sold: Widget C, Widget D, Widget A, Widget B
    Regions Covered: South, East, North, West
    Customer Age Range: 18 to 69 years
    Average Customer Satisfaction: 3.03 out of 5
    

Metadata: {'type': 'overall_summary'}


---
## Step 3: LLM Application Development

Build advanced data analysis functions and integrate with the RAG system for intelligent responses.

### Advanced Data Summary Functions

In [9]:
def get_sales_by_time_period(df: pd.DataFrame, period: str = 'Month') -> pd.DataFrame:
    """Aggregate sales metrics grouped by a time period.

    Parameters
    ----------
    df : pd.DataFrame
        Sales DataFrame with ``Month``, ``Quarter``, and ``Year`` columns.
    period : str
        One of ``'Month'``, ``'Quarter'``, or ``'Year'``.

    Returns
    -------
    pd.DataFrame
        Aggregated table with Total_Sales, Avg_Sale, Num_Transactions, Std_Dev.
    """
    period_col = {'Month': 'Month', 'Quarter': 'Quarter'}.get(period, 'Year')
    grouped = df.groupby(period_col)['Sales'].agg(['sum', 'mean', 'count', 'std'])
    grouped.columns = ['Total_Sales', 'Avg_Sale', 'Num_Transactions', 'Std_Dev']
    return grouped.round(2)


def get_product_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """Return per-product sales, satisfaction, and customer-age statistics."""
    product_stats = df.groupby('Product').agg({
        'Sales': ['sum', 'mean', 'median', 'std', 'count'],
        'Customer_Satisfaction': 'mean',
        'Customer_Age': 'mean'
    }).round(2)
    product_stats.columns = ['Total_Sales', 'Avg_Sale', 'Median_Sale', 'Std_Dev',
                             'Num_Transactions', 'Avg_Satisfaction', 'Avg_Customer_Age']
    return product_stats.sort_values('Total_Sales', ascending=False)


def get_regional_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """Return per-region sales, satisfaction, and customer-age statistics."""
    regional_stats = df.groupby('Region').agg({
        'Sales': ['sum', 'mean', 'count'],
        'Customer_Satisfaction': 'mean',
        'Customer_Age': 'mean'
    }).round(2)
    regional_stats.columns = ['Total_Sales', 'Avg_Sale', 'Num_Transactions',
                              'Avg_Satisfaction', 'Avg_Customer_Age']
    return regional_stats.sort_values('Total_Sales', ascending=False)


def get_customer_segmentation(df: pd.DataFrame) -> pd.DataFrame:
    """Segment customers into age-group × gender buckets with sales stats."""
    age_bins = [18, 30, 45, 60, 100]
    age_labels = ['Young (18-29)', 'Middle (30-44)', 'Senior (45-59)', 'Elder (60+)']
    df_copy = df.copy()
    df_copy['Age_Group'] = pd.cut(df_copy['Customer_Age'], bins=age_bins, labels=age_labels, right=False)

    segment_stats = df_copy.groupby(['Age_Group', 'Customer_Gender']).agg({
        'Sales': ['sum', 'mean', 'count'],
        'Customer_Satisfaction': 'mean'
    }).round(2)
    segment_stats.columns = ['Total_Sales', 'Avg_Sale', 'Num_Transactions', 'Avg_Satisfaction']
    return segment_stats


def get_statistical_measures(df: pd.DataFrame) -> dict:
    """Compute descriptive statistics and pairwise correlations for numeric columns."""
    stats = {
        'Sales': {
            'Mean': df['Sales'].mean(),
            'Median': df['Sales'].median(),
            'Std_Dev': df['Sales'].std(),
            'Min': df['Sales'].min(),
            'Max': df['Sales'].max(),
            'Skewness': df['Sales'].skew(),
            'Kurtosis': df['Sales'].kurtosis()
        },
        'Customer_Satisfaction': {
            'Mean': df['Customer_Satisfaction'].mean(),
            'Median': df['Customer_Satisfaction'].median(),
            'Std_Dev': df['Customer_Satisfaction'].std()
        },
        'Correlations': {
            'Sales_vs_Age': df['Sales'].corr(df['Customer_Age']),
            'Sales_vs_Satisfaction': df['Sales'].corr(df['Customer_Satisfaction']),
            'Age_vs_Satisfaction': df['Customer_Age'].corr(df['Customer_Satisfaction'])
        }
    }
    return stats

In [10]:
print("=== Sales by Month ===")
print(get_sales_by_time_period(df, 'Month').head())

print("\n=== Product Analysis ===")
print(get_product_analysis(df))

print("\n=== Regional Analysis ===")
print(get_regional_analysis(df))

=== Sales by Month ===
         Total_Sales  Avg_Sale  Num_Transactions  Std_Dev
Month                                                    
2022-01        18470    595.81                31   291.09
2022-02        15208    543.14                28   255.04
2022-03        14590    470.65                31   244.56
2022-04        13376    445.87                30   211.04
2022-05        16215    523.06                31   264.33

=== Product Analysis ===
          Total_Sales  Avg_Sale  Median_Sale  Std_Dev  Num_Transactions  \
Product                                                                   
Widget A       375235    572.00        582.0   257.58               656   
Widget B       346062    565.46        570.0   260.21               612   
Widget C       335069    540.43        541.5   263.98               620   
Widget D       326854    534.08        536.0   257.30               612   

          Avg_Satisfaction  Avg_Customer_Age  
Product                                       


### Custom Data Retriever

Build a custom retriever that extracts relevant statistics based on user queries.

In [11]:
class SalesDataRetriever:
    """Keyword-driven retriever that returns pre-computed analytics as context.

    Instead of embedding-only retrieval, this class inspects the user query for
    domain keywords (e.g. *product*, *region*, *trend*) and appends the matching
    aggregated statistics so the LLM has precise numbers to cite.
    """

    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df

    def retrieve_context(self, query: str) -> str:
        """Return a multi-section context string relevant to *query*."""
        query_lower = query.lower()
        context_parts = []
        
        context_parts.append(f"Dataset Overview: {len(self.df)} transactions, "
                           f"Total Sales: ${self.df['Sales'].sum():,.2f}")
        
        if any(word in query_lower for word in ['product', 'widget', 'item']):
            product_data = get_product_analysis(self.df)
            context_parts.append(f"\nProduct Performance:\n{product_data.to_string()}")
            
        if any(word in query_lower for word in ['region', 'north', 'south', 'east', 'west', 'geographic', 'location']):
            regional_data = get_regional_analysis(self.df)
            context_parts.append(f"\nRegional Performance:\n{regional_data.to_string()}")
            
        if any(word in query_lower for word in ['month', 'trend', 'time', 'period', 'quarter', 'year']):
            time_data = get_sales_by_time_period(self.df, 'Month')
            context_parts.append(f"\nMonthly Sales Trends:\n{time_data.to_string()}")
            
        if any(word in query_lower for word in ['customer', 'age', 'demographic', 'segment', 'gender']):
            segment_data = get_customer_segmentation(self.df)
            context_parts.append(f"\nCustomer Segmentation:\n{segment_data.to_string()}")
            
        if any(word in query_lower for word in ['satisfaction', 'happy', 'rating']):
            satisfaction_by_product = self.df.groupby('Product')['Customer_Satisfaction'].mean().round(2)
            satisfaction_by_region = self.df.groupby('Region')['Customer_Satisfaction'].mean().round(2)
            context_parts.append(f"\nSatisfaction by Product:\n{satisfaction_by_product.to_string()}")
            context_parts.append(f"\nSatisfaction by Region:\n{satisfaction_by_region.to_string()}")
            
        if any(word in query_lower for word in ['statistic', 'mean', 'median', 'average', 'std', 'correlation']):
            stats = get_statistical_measures(self.df)
            context_parts.append(f"\nStatistical Measures:\n{stats}")
            
        return "\n".join(context_parts)

data_retriever = SalesDataRetriever(df)
sample_context = data_retriever.retrieve_context("What are the sales by product?")
print("Sample Retrieved Context:")
print(sample_context[:1000])

Sample Retrieved Context:
Dataset Overview: 2500 transactions, Total Sales: $1,383,220.00

Product Performance:
          Total_Sales  Avg_Sale  Median_Sale  Std_Dev  Num_Transactions  Avg_Satisfaction  Avg_Customer_Age
Product                                                                                                    
Widget A       375235    572.00        582.0   257.58               656              3.03             43.39
Widget B       346062    565.46        570.0   260.21               612              2.99             43.67
Widget C       335069    540.43        541.5   263.98               620              3.01             42.71
Widget D       326854    534.08        536.0   257.30               612              3.07             43.57


---
## Step 4: Chain Prompts

Design prompt templates to ensure the LLM produces coherent and contextually relevant responses for different types of business queries.

In [12]:
# System prompt shared by all prompt templates so the LLM persona stays
# consistent regardless of which template is selected.
SYSTEM_PROMPT = """You are InsightForge, an AI-powered Business Intelligence Assistant specialized in analyzing sales data. 
Your role is to:
1. Provide accurate, data-driven insights based on the context provided
2. Identify trends, patterns, and anomalies in business data
3. Generate actionable recommendations for business improvement
4. Present information in a clear, professional manner

Always base your responses on the actual data provided in the context. If the data doesn't support a conclusion, say so clearly.
Format numbers appropriately (currency with $, percentages with %)."""

SALES_ANALYSIS_TEMPLATE = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """Based on the following sales data context, please analyze and answer the question.

DATA CONTEXT:
{context}

QUESTION: {question}

Provide a comprehensive analysis with specific numbers from the data.""")
])

TREND_ANALYSIS_TEMPLATE = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """Analyze the following time-series sales data and identify trends.

DATA CONTEXT:
{context}

QUESTION: {question}

Identify:
1. Overall trend direction (increasing/decreasing/stable)
2. Seasonal patterns if any
3. Notable peaks or dips
4. Recommendations based on trends""")
])

CUSTOMER_INSIGHT_TEMPLATE = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """Analyze customer demographics and behavior from the following data.

DATA CONTEXT:
{context}

QUESTION: {question}

Provide insights on:
1. Key customer segments
2. Purchasing patterns by demographic
3. Satisfaction levels
4. Recommendations for targeting customers""")
])

RECOMMENDATION_TEMPLATE = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """Based on the business data below, provide strategic recommendations.

DATA CONTEXT:
{context}

QUESTION: {question}

Structure your response as:
1. Key Findings (3-5 bullet points)
2. Strategic Recommendations (prioritized list)
3. Expected Impact""")
])

print("Prompt templates created successfully!")

Prompt templates created successfully!


In [13]:
def select_prompt_template(query: str) -> ChatPromptTemplate:
    """Choose the best prompt template by scanning *query* for domain keywords."""
    query_lower = query.lower()
    
    if any(word in query_lower for word in ['trend', 'time', 'month', 'quarter', 'year', 'growth', 'decline']):
        return TREND_ANALYSIS_TEMPLATE
    elif any(word in query_lower for word in ['customer', 'demographic', 'age', 'gender', 'segment', 'satisfaction']):
        return CUSTOMER_INSIGHT_TEMPLATE
    elif any(word in query_lower for word in ['recommend', 'suggest', 'improve', 'strategy', 'should']):
        return RECOMMENDATION_TEMPLATE
    else:
        return SALES_ANALYSIS_TEMPLATE

test_query = "What are the monthly sales trends?"
selected_template = select_prompt_template(test_query)
print(f"Query: {test_query}")
print(f"Selected template: {'TREND_ANALYSIS' if selected_template == TREND_ANALYSIS_TEMPLATE else 'SALES_ANALYSIS'}")

Query: What are the monthly sales trends?
Selected template: TREND_ANALYSIS


---
## Step 5: RAG System Setup

Implement the RAG (Retrieval-Augmented Generation) system with embeddings, vector store, and retrieval chain to enhance the LLM's ability to generate detailed and accurate responses.

### Create Embeddings and Vector Store

In [16]:
#!pip uninstall sentence-transformers -y && pip install sentence-transformers --force-reinstall --no-cache-dir

Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 21.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 15.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 16.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 16.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 16.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 16.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 14.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 16.4 MB/s eta 0:00:00
   

In [17]:
try:
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    logger.info("Embedding model loaded: %s", EMBEDDING_MODEL)
except ImportError as exc:
    raise ImportError(
        "sentence-transformers is required. "
        "Run: pip install sentence-transformers"
    ) from exc

vectorstore = FAISS.from_documents(knowledge_base_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": RAG_TOP_K})

print(f"Vector store created with {len(knowledge_base_docs)} documents")
print(f"Embedding dimension: {len(embeddings.embed_query('test'))}")

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [ ]:
test_query = "Which product has the highest sales?"
relevant_docs = retriever.invoke(test_query)
print(f"Retrieved {len(relevant_docs)} relevant documents for query: '{test_query}'")
print("\nTop relevant document:")
print(relevant_docs[0].page_content[:500])

### Build RAG Chain

In [ ]:
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs: list[Document]) -> str:
    """Concatenate retrieved Document page_content into one context block."""
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | SALES_ANALYSIS_TEMPLATE
    | llm
    | StrOutputParser()
)

print("RAG Chain created successfully!")

In [ ]:
response = rag_chain.invoke("Which product generates the most revenue and why?")
print("=== RAG Chain Response ===")
print(response)

### Hybrid RAG System (Vector Store + Custom Retriever)

In [ ]:
class HybridRAGSystem:
    """Combines vector store retrieval with custom data analysis."""
    
    def __init__(self, vectorstore, data_retriever, llm):
        self.vectorstore = vectorstore
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        self.data_retriever = data_retriever
        self.llm = llm
        
    def get_combined_context(self, query):
        """Combine vector store documents with custom retrieved data."""
        vector_docs = self.retriever.invoke(query)
        vector_context = "\n\n".join(doc.page_content for doc in vector_docs)
        
        custom_context = self.data_retriever.retrieve_context(query)
        
        combined = f"=== KNOWLEDGE BASE INSIGHTS ===\n{vector_context}\n\n"
        combined += f"=== REAL-TIME DATA ANALYSIS ===\n{custom_context}"
        
        return combined
    
    def query(self, question, template=None):
        """Process a question through the hybrid RAG system."""
        if template is None:
            template = select_prompt_template(question)
            
        context = self.get_combined_context(question)
        
        chain = template | self.llm | StrOutputParser()
        response = chain.invoke({"context": context, "question": question})
        
        return response

hybrid_rag = HybridRAGSystem(vectorstore, data_retriever, llm)
print("Hybrid RAG System initialized!")

In [ ]:
response = hybrid_rag.query("Compare the performance of different regions and identify the best performing one.")
print("=== Hybrid RAG Response ===")
print(response)

---
## Step 6: Memory Integration

Integrate memory systems to enable the model to retain and use contextual information from previous interactions, thereby improving the relevance of responses.

In [ ]:
# ConversationBufferMemory keeps the full chat history in memory so the
# retrieval chain can use prior turns for follow-up questions.
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False,
)

print("Conversational chain with memory created!")

In [ ]:
class InsightForgeAssistant:
    """Full-featured BI Assistant combining vector retrieval, keyword
    retrieval, and conversation memory.

    The assistant merges FAISS vector-store results with keyword-driven
    ``SalesDataRetriever`` context and maintains a sliding window of
    the last six conversation turns.
    """

    def __init__(self, df: pd.DataFrame, vectorstore, llm) -> None:
        self.df = df
        self.vectorstore = vectorstore
        self.llm = llm
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        self.data_retriever = SalesDataRetriever(df)
        self.conversation_history: list[dict[str, str]] = []

        self.memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
        )

    def get_context(self, query: str) -> str:
        """Merge vector-store and keyword-retriever context for *query*."""
        vector_docs = self.retriever.invoke(query)
        vector_context = "\n".join(doc.page_content for doc in vector_docs)
        data_context = self.data_retriever.retrieve_context(query)
        return f"{vector_context}\n\n{data_context}"

    def chat(self, user_message: str) -> str:
        """Process user message and return response with memory."""
        self.conversation_history.append({"role": "user", "content": user_message})
        
        context = self.get_context(user_message)
        
        history_text = "\n".join([
            f"{msg['role'].upper()}: {msg['content']}" 
            for msg in self.conversation_history[-6:-1]
        ])
        
        template = select_prompt_template(user_message)
        
        prompt_with_history = ChatPromptTemplate.from_messages([
            ("system", SYSTEM_PROMPT + "\n\nPrevious conversation:\n" + history_text),
            ("human", """Based on the following data context, answer the question.

DATA CONTEXT:
{context}

QUESTION: {question}""")
        ])
        
        chain = prompt_with_history | self.llm | StrOutputParser()
        response = chain.invoke({"context": context, "question": user_message})
        
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return response
    
    def reset_conversation(self) -> None:
        """Clear conversation history."""
        self.conversation_history = []
        print("Conversation history cleared.")

    def get_conversation_summary(self) -> str:
        """Return a one-line summary of the current conversation length."""
        if not self.conversation_history:
            return "No conversation history."
        return f"Total messages: {len(self.conversation_history)}"

assistant = InsightForgeAssistant(df, vectorstore, llm)
print("InsightForge Assistant initialized with memory!")

In [ ]:
print("=== Conversation with Memory Demo ===\n")

response1 = assistant.chat("What is the total sales for Widget A?")
print(f"Q1: What is the total sales for Widget A?")
print(f"A1: {response1}\n")

response2 = assistant.chat("How does that compare to Widget B?")
print(f"Q2: How does that compare to Widget B?")
print(f"A2: {response2}\n")

response3 = assistant.chat("Which region has the highest sales for the product we discussed first?")
print(f"Q3: Which region has the highest sales for the product we discussed first?")
print(f"A3: {response3}")

print(f"\n{assistant.get_conversation_summary()}")

---
# Part 2: LLMOps (Model Evaluation, Monitoring, and User Interface)

## Step 7a: Model Evaluation

Apply QAEvalChain to assess the model's performance and accuracy using predefined question-answer pairs.

In [ ]:
# Ground-truth metrics computed directly from the DataFrame so the
# evaluation QA pairs stay in sync with the actual data.
total_sales = df['Sales'].sum()
avg_sale = df['Sales'].mean()
product_sales = df.groupby('Product')['Sales'].sum()
best_product = product_sales.idxmax()
region_sales = df.groupby('Region')['Sales'].sum()
best_region = region_sales.idxmax()
avg_satisfaction = df['Customer_Satisfaction'].mean()

eval_qa_pairs = [
    {
        "question": "What is the total sales revenue in the dataset?",
        "answer": f"The total sales revenue is approximately ${total_sales:,.0f}"
    },
    {
        "question": "Which product has the highest total sales?",
        "answer": f"{best_product} has the highest total sales with ${product_sales[best_product]:,.0f}"
    },
    {
        "question": "Which region performs best in terms of total sales?",
        "answer": f"The {best_region} region performs best with total sales of ${region_sales[best_region]:,.0f}"
    },
    {
        "question": "What is the average customer satisfaction rating?",
        "answer": f"The average customer satisfaction rating is {avg_satisfaction:.2f} out of 5"
    },
    {
        "question": "How many products are sold in the dataset?",
        "answer": f"There are {df['Product'].nunique()} different products: {', '.join(df['Product'].unique())}"
    }
]

print(f"Created {len(eval_qa_pairs)} evaluation QA pairs")

In [ ]:
assistant.reset_conversation()

predictions = []
for qa in eval_qa_pairs:
    response = assistant.chat(qa["question"])
    predictions.append({
        "question": qa["question"],
        "answer": qa["answer"],
        "result": response
    })
    assistant.reset_conversation()
    
print("Generated predictions for evaluation")

In [ ]:
eval_chain = QAEvalChain.from_llm(llm)

graded_outputs = eval_chain.evaluate(
    eval_qa_pairs,
    predictions,
    question_key="question",
    answer_key="answer",
    prediction_key="result"
)

print("=== Model Evaluation Results ===\n")
correct = 0
for i, (qa, pred, grade) in enumerate(zip(eval_qa_pairs, predictions, graded_outputs)):
    result = grade.get('results', grade.get('text', 'N/A'))
    is_correct = 'CORRECT' in str(result).upper()
    correct += 1 if is_correct else 0
    
    print(f"Q{i+1}: {qa['question']}")
    print(f"Expected: {qa['answer'][:100]}...")
    print(f"Got: {pred['result'][:100]}...")
    print(f"Grade: {result}")
    print("-" * 50)

print(f"\n=== Overall Score: {correct}/{len(eval_qa_pairs)} ({100*correct/len(eval_qa_pairs):.1f}%) ===")

---
## Step 7b: Data Visualization

Create various plots and visualizations to present insights including:
- Sales trends over time
- Product performance comparisons
- Regional analysis
- Customer demographics and segmentation

In [ ]:
def plot_sales_trends(df: pd.DataFrame) -> plt.Figure:
    """Create a 2×2 grid of sales trend charts (monthly line, quarterly bar,
    monthly average area, monthly transaction count bar)."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    monthly_sales = df.groupby('Month')['Sales'].sum()
    ax1 = axes[0, 0]
    monthly_sales.plot(kind='line', marker='o', ax=ax1, color='steelblue', linewidth=2)
    ax1.set_title('Monthly Sales Trend', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Month')
    ax1.set_ylabel('Total Sales ($)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    
    quarterly_sales = df.groupby('Quarter')['Sales'].sum()
    ax2 = axes[0, 1]
    quarterly_sales.plot(kind='bar', ax=ax2, color='coral')
    ax2.set_title('Quarterly Sales', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Quarter')
    ax2.set_ylabel('Total Sales ($)')
    ax2.tick_params(axis='x', rotation=0)
    
    monthly_avg = df.groupby('Month')['Sales'].mean()
    ax3 = axes[1, 0]
    monthly_avg.plot(kind='area', ax=ax3, alpha=0.7, color='mediumseagreen')
    ax3.set_title('Monthly Average Sale Value', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Month')
    ax3.set_ylabel('Average Sale ($)')
    ax3.tick_params(axis='x', rotation=45)
    
    monthly_count = df.groupby('Month')['Sales'].count()
    ax4 = axes[1, 1]
    monthly_count.plot(kind='bar', ax=ax4, color='mediumpurple')
    ax4.set_title('Monthly Transaction Count', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Month')
    ax4.set_ylabel('Number of Transactions')
    ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    return fig

fig = plot_sales_trends(df)
plt.show()

In [ ]:
def plot_product_performance(df: pd.DataFrame) -> plt.Figure:
    """Create a 2×2 grid comparing product sales, share, satisfaction, and
    monthly trends."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    
    product_sales = df.groupby('Product')['Sales'].sum().sort_values(ascending=True)
    ax1 = axes[0, 0]
    product_sales.plot(kind='barh', ax=ax1, color=colors)
    ax1.set_title('Total Sales by Product', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Total Sales ($)')
    for i, v in enumerate(product_sales.values):
        ax1.text(v + 1000, i, f'${v:,.0f}', va='center')
    
    ax2 = axes[0, 1]
    product_sales.plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=colors, startangle=90)
    ax2.set_title('Sales Distribution by Product', fontsize=12, fontweight='bold')
    ax2.set_ylabel('')
    
    product_satisfaction = df.groupby('Product')['Customer_Satisfaction'].mean().sort_values()
    ax3 = axes[1, 0]
    bars = ax3.bar(product_satisfaction.index, product_satisfaction.values, color=colors)
    ax3.set_title('Average Customer Satisfaction by Product', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Satisfaction Rating')
    ax3.set_ylim(0, 5)
    ax3.axhline(y=df['Customer_Satisfaction'].mean(), color='red', linestyle='--', label='Overall Avg')
    ax3.legend()
    
    ax4 = axes[1, 1]
    for product in df['Product'].unique():
        product_monthly = df[df['Product'] == product].groupby('Month')['Sales'].sum()
        ax4.plot(range(len(product_monthly)), product_monthly.values, marker='o', label=product)
    ax4.set_title('Monthly Sales by Product', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Month')
    ax4.set_ylabel('Sales ($)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

fig = plot_product_performance(df)
plt.show()

In [ ]:
def plot_regional_analysis(df: pd.DataFrame) -> plt.Figure:
    """Create a 2×2 grid of regional performance charts (bar, grouped bar,
    satisfaction, and regional metrics comparison)."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
    
    region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
    ax1 = axes[0, 0]
    bars = ax1.bar(region_sales.index, region_sales.values, color=colors)
    ax1.set_title('Total Sales by Region', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Total Sales ($)')
    for bar, val in zip(bars, region_sales.values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, 
                f'${val:,.0f}', ha='center', fontsize=9)
    
    region_product = df.pivot_table(values='Sales', index='Region', columns='Product', aggfunc='sum')
    ax2 = axes[0, 1]
    region_product.plot(kind='bar', ax=ax2, width=0.8)
    ax2.set_title('Sales by Region and Product', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Sales ($)')
    ax2.tick_params(axis='x', rotation=0)
    ax2.legend(title='Product', bbox_to_anchor=(1.02, 1))
    
    region_satisfaction = df.groupby('Region')['Customer_Satisfaction'].mean()
    ax3 = axes[1, 0]
    ax3.bar(region_satisfaction.index, region_satisfaction.values, color=colors)
    ax3.set_title('Average Satisfaction by Region', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Satisfaction Rating')
    ax3.set_ylim(0, 5)
    ax3.axhline(y=df['Customer_Satisfaction'].mean(), color='red', linestyle='--', alpha=0.7)
    
    region_stats = df.groupby('Region').agg({
        'Sales': 'mean',
        'Customer_Age': 'mean'
    })
    ax4 = axes[1, 1]
    x = range(len(region_stats))
    width = 0.35
    bars1 = ax4.bar([i - width/2 for i in x], region_stats['Sales']/100, width, label='Avg Sale (×100)', color='steelblue')
    bars2 = ax4.bar([i + width/2 for i in x], region_stats['Customer_Age'], width, label='Avg Age', color='coral')
    ax4.set_title('Regional Metrics Comparison', fontsize=12, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels(region_stats.index)
    ax4.legend()
    
    plt.tight_layout()
    return fig

fig = plot_regional_analysis(df)
plt.show()

In [ ]:
def plot_customer_demographics(df: pd.DataFrame) -> plt.Figure:
    """Create a 2×3 grid covering age histogram, gender pie, gender sales,
    age-group sales, satisfaction box, and age-group × gender bars."""
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    ax1 = axes[0, 0]
    ax1.hist(df['Customer_Age'], bins=20, color='steelblue', edgecolor='white', alpha=0.7)
    ax1.axvline(df['Customer_Age'].mean(), color='red', linestyle='--', label=f'Mean: {df["Customer_Age"].mean():.1f}')
    ax1.set_title('Customer Age Distribution', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Age')
    ax1.set_ylabel('Frequency')
    ax1.legend()
    
    gender_counts = df['Customer_Gender'].value_counts()
    ax2 = axes[0, 1]
    ax2.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', 
            colors=['#3498DB', '#E74C3C'], startangle=90, explode=[0.02, 0.02])
    ax2.set_title('Gender Distribution', fontsize=12, fontweight='bold')
    
    gender_sales = df.groupby('Customer_Gender')['Sales'].sum()
    ax3 = axes[0, 2]
    ax3.bar(gender_sales.index, gender_sales.values, color=['#3498DB', '#E74C3C'])
    ax3.set_title('Total Sales by Gender', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Total Sales ($)')
    
    age_bins = [18, 30, 45, 60, 100]
    age_labels = ['18-29', '30-44', '45-59', '60+']
    df_copy = df.copy()
    df_copy['Age_Group'] = pd.cut(df_copy['Customer_Age'], bins=age_bins, labels=age_labels, right=False)
    
    age_sales = df_copy.groupby('Age_Group')['Sales'].sum()
    ax4 = axes[1, 0]
    ax4.bar(age_sales.index, age_sales.values, color=['#2ECC71', '#3498DB', '#9B59B6', '#E74C3C'])
    ax4.set_title('Sales by Age Group', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Age Group')
    ax4.set_ylabel('Total Sales ($)')
    
    ax5 = axes[1, 1]
    df.boxplot(column='Customer_Satisfaction', by='Customer_Gender', ax=ax5)
    ax5.set_title('Satisfaction by Gender', fontsize=12, fontweight='bold')
    ax5.set_xlabel('Gender')
    ax5.set_ylabel('Satisfaction Rating')
    plt.suptitle('')
    
    age_gender_sales = df_copy.pivot_table(values='Sales', index='Age_Group', columns='Customer_Gender', aggfunc='sum')
    ax6 = axes[1, 2]
    age_gender_sales.plot(kind='bar', ax=ax6, color=['#E74C3C', '#3498DB'])
    ax6.set_title('Sales by Age Group and Gender', fontsize=12, fontweight='bold')
    ax6.set_xlabel('Age Group')
    ax6.set_ylabel('Total Sales ($)')
    ax6.tick_params(axis='x', rotation=0)
    ax6.legend(title='Gender')
    
    plt.tight_layout()
    return fig

fig = plot_customer_demographics(df)
plt.show()

In [ ]:
def plot_satisfaction_analysis(df: pd.DataFrame) -> plt.Figure:
    """Create a 2×2 grid of satisfaction analysis charts (histogram,
    sales-vs-satisfaction scatter, product box, and region box)."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax1 = axes[0, 0]
    ax1.hist(df['Customer_Satisfaction'], bins=20, color='mediumseagreen', edgecolor='white', alpha=0.7)
    ax1.axvline(df['Customer_Satisfaction'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df["Customer_Satisfaction"].mean():.2f}')
    ax1.set_title('Satisfaction Score Distribution', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Satisfaction Rating')
    ax1.set_ylabel('Frequency')
    ax1.legend()
    
    ax2 = axes[0, 1]
    ax2.scatter(df['Sales'], df['Customer_Satisfaction'], alpha=0.5, c='steelblue')
    ax2.set_title('Sales vs Satisfaction', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Sales ($)')
    ax2.set_ylabel('Satisfaction Rating')
    z = np.polyfit(df['Sales'], df['Customer_Satisfaction'], 1)
    p = np.poly1d(z)
    ax2.plot(df['Sales'].sort_values(), p(df['Sales'].sort_values()), "r--", alpha=0.8, 
             label=f'Corr: {df["Sales"].corr(df["Customer_Satisfaction"]):.3f}')
    ax2.legend()
    
    ax3 = axes[1, 0]
    df.boxplot(column='Customer_Satisfaction', by='Product', ax=ax3)
    ax3.set_title('Satisfaction by Product', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Product')
    ax3.set_ylabel('Satisfaction Rating')
    plt.suptitle('')
    
    ax4 = axes[1, 1]
    df.boxplot(column='Customer_Satisfaction', by='Region', ax=ax4)
    ax4.set_title('Satisfaction by Region', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Region')
    ax4.set_ylabel('Satisfaction Rating')
    plt.suptitle('')
    
    plt.tight_layout()
    return fig

fig = plot_satisfaction_analysis(df)
plt.show()

---
## Step 7c: Streamlit User Interface

Develop an intuitive user interface using Streamlit, allowing users to interact with the AI assistant and access visualizations and insights.

The code below creates a complete Streamlit application. To run it:
1. Export this notebook as a Python script (.py)
2. Remove all print statements and display calls
3. Run: `streamlit run your_script_name.py`

In [ ]:
STREAMLIT_CODE = '''
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()

st.set_page_config(
    page_title="InsightForge BI Assistant",
    page_icon="📊",
    layout="wide"
)

st.title("📊 InsightForge: AI-Powered Business Intelligence Assistant")
st.markdown("---")

@st.cache_data
def load_data():
    df = pd.read_csv('sales_data.csv')
    df['Date'] = pd.to_datetime(df['Date'])
    df['Month'] = df['Date'].dt.to_period('M').astype(str)
    return df

@st.cache_resource
def initialize_rag_system(_df):
    llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    documents = []
    overall = f"""Dataset: {len(_df)} transactions, Total Sales: ${_df['Sales'].sum():,.2f}, 
    Products: {', '.join(_df['Product'].unique())}, Regions: {', '.join(_df['Region'].unique())}"""
    documents.append(Document(page_content=overall, metadata={"type": "overview"}))
    
    for product in _df['Product'].unique():
        pdf = _df[_df['Product'] == product]
        doc = f"Product {product}: Total ${pdf['Sales'].sum():,.2f}, Avg ${pdf['Sales'].mean():.2f}, Satisfaction {pdf['Customer_Satisfaction'].mean():.2f}"
        documents.append(Document(page_content=doc, metadata={"type": "product"}))
    
    for region in _df['Region'].unique():
        rdf = _df[_df['Region'] == region]
        doc = f"Region {region}: Total ${rdf['Sales'].sum():,.2f}, Avg ${rdf['Sales'].mean():.2f}, Satisfaction {rdf['Customer_Satisfaction'].mean():.2f}"
        documents.append(Document(page_content=doc, metadata={"type": "region"}))
    
    vectorstore = FAISS.from_documents(documents, embeddings)
    return llm, vectorstore

df = load_data()
llm, vectorstore = initialize_rag_system(df)

st.sidebar.header("📈 Dashboard Controls")
view = st.sidebar.selectbox("Select View", ["AI Assistant", "Sales Overview", "Product Analysis", "Regional Analysis", "Customer Demographics"])

if view == "AI Assistant":
    st.header("🤖 Chat with InsightForge")
    
    if "messages" not in st.session_state:
        st.session_state.messages = []
    
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])
    
    if prompt := st.chat_input("Ask about your sales data..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)
        
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        docs = retriever.invoke(prompt)
        context = "\\n".join([d.page_content for d in docs])
        
        template = ChatPromptTemplate.from_messages([
            ("system", "You are InsightForge, a BI assistant. Use the data context to answer questions accurately."),
            ("human", "Context: {context}\\n\\nQuestion: {question}")
        ])
        
        chain = template | llm | StrOutputParser()
        response = chain.invoke({"context": context, "question": prompt})
        
        st.session_state.messages.append({"role": "assistant", "content": response})
        with st.chat_message("assistant"):
            st.markdown(response)

elif view == "Sales Overview":
    st.header("📊 Sales Overview Dashboard")
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Sales", f"${df['Sales'].sum():,.0f}")
    col2.metric("Avg Transaction", f"${df['Sales'].mean():.0f}")
    col3.metric("Total Transactions", f"{len(df):,}")
    col4.metric("Avg Satisfaction", f"{df['Customer_Satisfaction'].mean():.2f}/5")
    
    fig, ax = plt.subplots(figsize=(10, 4))
    monthly = df.groupby('Month')['Sales'].sum()
    ax.plot(range(len(monthly)), monthly.values, marker='o')
    ax.set_title('Monthly Sales Trend')
    ax.set_xlabel('Month')
    ax.set_ylabel('Sales ($)')
    st.pyplot(fig)

elif view == "Product Analysis":
    st.header("📦 Product Performance")
    product_data = df.groupby('Product').agg({'Sales': ['sum', 'mean', 'count'], 'Customer_Satisfaction': 'mean'}).round(2)
    product_data.columns = ['Total Sales', 'Avg Sale', 'Transactions', 'Satisfaction']
    st.dataframe(product_data)
    
    fig, ax = plt.subplots()
    df.groupby('Product')['Sales'].sum().plot(kind='bar', ax=ax)
    ax.set_title('Total Sales by Product')
    st.pyplot(fig)

elif view == "Regional Analysis":
    st.header("🗺️ Regional Performance")
    region_data = df.groupby('Region').agg({'Sales': ['sum', 'mean'], 'Customer_Satisfaction': 'mean'}).round(2)
    region_data.columns = ['Total Sales', 'Avg Sale', 'Satisfaction']
    st.dataframe(region_data)
    
    fig, ax = plt.subplots()
    df.groupby('Region')['Sales'].sum().plot(kind='pie', ax=ax, autopct='%1.1f%%')
    ax.set_title('Sales Distribution by Region')
    st.pyplot(fig)

else:
    st.header("👥 Customer Demographics")
    col1, col2 = st.columns(2)
    with col1:
        fig, ax = plt.subplots()
        ax.hist(df['Customer_Age'], bins=20, edgecolor='white')
        ax.set_title('Age Distribution')
        st.pyplot(fig)
    with col2:
        fig, ax = plt.subplots()
        df['Customer_Gender'].value_counts().plot(kind='pie', ax=ax, autopct='%1.1f%%')
        ax.set_title('Gender Distribution')
        st.pyplot(fig)

st.sidebar.markdown("---")
st.sidebar.info("💡 Ask questions like:\\n- What product has highest sales?\\n- Compare regional performance\\n- Show customer demographics")
'''

print("Streamlit code ready for deployment!")
print("\\nTo run the Streamlit app:")
print("1. Save this code to a file (e.g., 'insightforge_app.py')")
print("2. Run: streamlit run insightforge_app.py")

In [ ]:
with open('insightforge_app.py', 'w') as f:
    f.write(STREAMLIT_CODE)
print("Streamlit app saved to 'insightforge_app.py'")

---
# Summary and Conclusion

## Project Implementation Complete

This notebook successfully implements the **InsightForge AI-Powered Business Intelligence Assistant** with all required components:

### Part 1: AI-Powered Business Intelligence Assistant
1. **Data Preparation** - Loaded and explored 2,500 sales transactions
2. **Knowledge Base Creation** - Created structured documents for RAG retrieval
3. **LLM Application Development** - Built advanced analysis functions with custom retrievers
4. **Chain Prompts** - Designed specialized prompts for different query types
5. **RAG System Setup** - Implemented FAISS vector store with HuggingFace embeddings
6. **Memory Integration** - Added conversation memory for contextual responses

### Part 2: LLMOps
7. **Model Evaluation** - Used QAEvalChain to assess accuracy
8. **Data Visualization** - Created comprehensive charts and graphs
9. **Streamlit UI** - Built interactive dashboard for end users

## Key Technologies Used
- **LangChain** for LLM orchestration
- **FAISS** for vector storage
- **HuggingFace** embeddings
- **OpenAI GPT** for generation
- **Pandas/Matplotlib/Seaborn** for data analysis
- **Streamlit** for user interface

## Running the Streamlit App
```bash
streamlit run insightforge_app.py
```